# 05 — MuJoCo Simulation

Physics simulation with the MuJoCo backend. The `Simulation` class is both
a `SimEngine` implementation and a Strands `AgentTool`.

**Requires**: `pip install "strands-robots[sim-mujoco]"`

In [ ]:
from strands_robots.simulation import create_simulation, list_backends

print(f"backends: {list_backends()}")

## World Lifecycle

In [ ]:
sim = create_simulation("mujoco")  # or "mj", "mjc", "mjx" — all aliases

result = sim.create_world(
    timestep=0.002,        # 500 Hz physics
    gravity=[0, 0, -9.81],
    ground_plane=True,
)
print(result["content"][0]["text"])

## Adding Robots

Resolved from the model registry — checks local assets, then Menagerie.

In [ ]:
result = sim.add_robot(
    name="so100",
    data_config="so100",
    position=[0, 0, 0],
)
print(result["content"][0]["text"])

## Adding Objects

In [ ]:
sim.add_object("red_cube", shape="box", position=[0.3, 0, 0.05],
               size=[0.03, 0.03, 0.03], color=[1, 0, 0, 1], mass=0.05)
sim.add_object("blue_sphere", shape="sphere", position=[0.2, 0.1, 0.05],
               size=[0.02], color=[0, 0, 1, 1], mass=0.03)
sim.add_object("table", shape="box", position=[0.3, 0, -0.02],
               size=[0.3, 0.3, 0.02], color=[0.6, 0.4, 0.2, 1], is_static=True)
print("objects added ✅")

## Stepping & Observation

In [ ]:
sim.step(n_steps=100)

obs = sim.get_observation("so100")

# Separate scalar joints from image arrays
joints = {k: v for k, v in obs.items() if not hasattr(v, 'shape') or getattr(v, 'ndim', 0) < 2}
images = {k: v for k, v in obs.items() if hasattr(v, 'shape') and getattr(v, 'ndim', 0) >= 2}

print("joints:")
for k, v in joints.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
print(f"images: { {k: v.shape for k, v in images.items()} }")

## Sending Actions

In [ ]:
import numpy as np

action = {k: float(np.sin(0.5)) for k in joints}
sim.send_action(action, robot_name="so100")

for _ in range(50):
    sim.step(n_steps=1)

new_obs = sim.get_observation("so100")
for k in joints:
    actual = new_obs.get(k, 0.0)
    print(f"  {k}: target={action[k]:.4f} → actual={actual:.4f}")

## Rendering

In [ ]:
result = sim.render(camera_name="default", width=640, height=480)
if "image" in result:
    img = result["image"]
    print(f"image: {img.shape} {img.dtype}")
    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(8, 6))
        plt.imshow(img)
        plt.axis("off")
        plt.show()
    except ImportError:
        print("(install matplotlib for inline display)")
else:
    print("rendering unavailable (headless without EGL/OSMesa)")

## Physics Features (PhysicsMixin)

In [ ]:
# Save/restore state checkpoints
sim.save_state("checkpoint_1")
sim.step(n_steps=200)
sim.load_state("checkpoint_1")
print("state restored ✅")

In [ ]:
sim.destroy()
print("world destroyed ✅")